<a href="https://colab.research.google.com/github/swaranjalilamkane/Agentic-AI-Loan-Risk-Assesment/blob/main/data_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kagglehub ucimlrepo


In [2]:
"""
SAAi Project - Data Validation Module
Team: Swaranjali Lamkane & Asha Subramaniyan
Course: Foundations of Artificial Intelligence

Runs all validation checks on:
  - Lending Club dataset  (via kagglehub)
  - German Credit dataset (via ucimlrepo)

Sections:
  4.1 Schema Validation
  4.2 Completeness & Missingness Checks
  4.3 Consistency Checks
  4.4 Fairness-Aware Data Checks
"""

import os
import glob
import logging
import numpy as np
import pandas as pd
import kagglehub
from ucimlrepo import fetch_ucirepo

import sys

# Force logger output to stdout so Colab displays it inline
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
    logger.addHandler(handler)
logger.propagate = False


# ════════════════════════════════════════════════════════
#  DATASET LOADERS
# ════════════════════════════════════════════════════════

def load_lending_club(nrows=50000) -> pd.DataFrame:
    """Download Lending Club via kagglehub and return prepared DataFrame."""
    path = kagglehub.dataset_download("wordsforthewise/lending-club")
    print("Path to dataset files:", path)

    # Find all files (not directories) recursively
    all_files = [
        f for f in glob.glob(os.path.join(path, "**", "*"), recursive=True)
        if os.path.isfile(f)
    ]
    print(f"Files found: {all_files}")

    accepted = [f for f in all_files if "accepted" in os.path.basename(f).lower()]
    target   = accepted[0] if accepted else sorted(all_files, key=os.path.getsize, reverse=True)[0]
    print(f"Loading: {target}")

    df = pd.read_csv(target, low_memory=False, nrows=nrows)
    df.dropna(how="all", inplace=True)
    print(f"Loaded {len(df):,} records, {df.shape[1]} columns.\n")
    return _prepare_lending_club(df)


def load_german_credit() -> pd.DataFrame:
    """Fetch German Credit dataset from UCI and return prepared DataFrame."""
    print("Fetching German Credit dataset from UCI...")
    ds = fetch_ucirepo(id=144)

    print("\n── Metadata ──")
    print(ds.metadata)
    print("\n── Variable Information ──")
    print(ds.variables)

    X = ds.data.features.copy()
    y = ds.data.targets.copy()

    # target column is 'class': 1 = good credit, 2 = bad credit → map to binary 0/1
    target_col = y.columns[0]
    df = pd.concat([X, y], axis=1)
    df.rename(columns={target_col: "default"}, inplace=True)
    df["default"] = df["default"].map({1: 0, 2: 1})   # 0 = good, 1 = bad (default)

    print(f"\nLoaded {len(df):,} records, {df.shape[1]} columns.\n")
    return _prepare_german_credit(df)


# ════════════════════════════════════════════════════════
#  COLUMN PREPARATION
# ════════════════════════════════════════════════════════

# ── Lending Club ──────────────────────────────────────

LC_COLUMN_MAP = {
    "annual_inc":       "income",
    "loan_amnt":        "loan_amount",
    "earliest_cr_line": "credit_history_length",
    "emp_title":        "employment_status",
    "dti":              "dti",
    "fico_range_low":   "fico_score",
    "installment":      "monthly_payment",
    "term":             "loan_term",
    "int_rate":         "interest_rate",
    "loan_status":      "default",
}

def _prepare_lending_club(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={k: v for k, v in LC_COLUMN_MAP.items() if k in df.columns})

    # earliest_cr_line → years of credit history
    if "credit_history_length" in df.columns:
        df["credit_history_length"] = pd.to_datetime(
            df["credit_history_length"], format="%b-%Y", errors="coerce"
        )
        df["credit_history_length"] = (
            (pd.Timestamp.now() - df["credit_history_length"]).dt.days / 365.25
        ).round(1)

    # ' 36 months' → 36
    if "loan_term" in df.columns:
        df["loan_term"] = df["loan_term"].astype(str).str.extract(r"(\d+)").astype(float)

    # '10.65%' → 10.65
    if "interest_rate" in df.columns:
        df["interest_rate"] = df["interest_rate"].astype(str).str.replace("%", "").astype(float)

    # binary default label
    if "default" in df.columns:
        df["default"] = df["default"].map(
            lambda x: 1 if str(x).strip() in ("Charged Off", "Default")
                      else (0 if str(x).strip() == "Fully Paid" else np.nan)
        )

    # simplify employment_status
    if "employment_status" in df.columns:
        df["employment_status"] = df["employment_status"].apply(
            lambda x: "unemployed" if pd.isnull(x) or str(x).strip() == ""
                      else "employed"
        )

    # DTI in LC is percentage (e.g. 18.5) → normalise to 0–1
    if "dti" in df.columns:
        df["dti"] = df["dti"] / 100

    df["_source"] = "lending_club"
    return df


# ── German Credit ─────────────────────────────────────

GC_COLUMN_MAP = {
    "credit_amount":        "loan_amount",
    "duration":             "loan_term",          # months
    "age":                  "age",
    "foreign_worker":       "foreign_worker",
    "personal_status_sex":  "personal_status_sex",
    "employment":           "employment_status",   # UCI ucimlrepo uses 'employment'
    "checking_status":      "checking_status",
    "savings_status":       "savings_status",
    "installment_commitment": "installment_rate",
    "housing":              "housing",
    "job":                  "job",
}

def _prepare_german_credit(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={k: v for k, v in GC_COLUMN_MAP.items() if k in df.columns})

    # Derive income proxy: German Credit has no direct income field.
    # We approximate it as loan_amount / (loan_term / 12) as a rough monthly obligation ratio.
    if "loan_amount" in df.columns and "loan_term" in df.columns:
        df["income"] = np.nan   # not available — mark explicitly

    # employment_status categories in UCI German Credit (e.g. A71–A75 or text from ucimlrepo)
    if "employment_status" in df.columns:
        unemp_vals = ["A71", "unemployed", "unemployed/unskilled non-resident"]
        df["employment_status"] = df["employment_status"].apply(
            lambda x: "unemployed" if str(x).strip() in unemp_vals else "employed"
        )

    # DTI not present — mark as NaN
    df["dti"]              = np.nan
    df["fico_score"]       = np.nan
    df["monthly_payment"]  = np.nan
    df["interest_rate"]    = np.nan

    # Derive credit_history_length from loan_term (months) as a proxy
    if "loan_term" in df.columns:
        df["credit_history_length"] = df["loan_term"] / 12.0

    # Derive age_group for fairness checks
    if "age" in df.columns:
        df["age_group"] = pd.cut(
            df["age"],
            bins=[18, 25, 35, 50, 75],
            labels=["18-25", "26-35", "36-50", "51+"]
        ).astype(str)

    df["_source"] = "german_credit"
    return df


# ════════════════════════════════════════════════════════
#  SHARED VALIDATION CHECKS
# ════════════════════════════════════════════════════════

# ── 4.1 Schema Validation ─────────────────────────────

REQUIRED_FIELDS = ["loan_amount", "employment_status"]   # common to both datasets

NUMERIC_FIELDS  = ["income", "loan_amount", "dti", "fico_score",
                   "credit_history_length", "monthly_payment", "loan_term", "interest_rate"]

CATEGORICAL_FIELDS = {
    "employment_status": ["employed", "unemployed", "self-employed", "retired"]
}

RANGE_BOUNDS = {
    "income":      (0,   None),
    "loan_amount": (0,   None),
    "dti":         (0,   1),
    "fico_score":  (300, 850),
    "age":         (18,  100),
    "loan_term":   (1,   360),
}


def check_required_fields(df: pd.DataFrame, source: str) -> pd.DataFrame:
    present = [f for f in REQUIRED_FIELDS if f in df.columns]
    mask    = df[present].isnull().any(axis=1)
    if mask.any():
        for idx, row in df[mask].head(5).iterrows():
            cols = [f for f in present if pd.isnull(row[f])]
            logger.warning(f"  [{source}] Record {idx} rejected — missing: {cols}")
        if mask.sum() > 5:
            logger.warning(f"  [{source}] ... and {mask.sum()-5} more.")
    logger.info(f"[4.1][{source}] Required fields: {mask.sum()} records rejected.")
    return df[~mask].copy()


def enforce_data_types(df: pd.DataFrame, source: str) -> pd.DataFrame:
    valid_idx = df.index.tolist()
    for col in NUMERIC_FIELDS:
        if col not in df.columns:
            continue
        coerced = pd.to_numeric(df[col], errors="coerce")
        bad = coerced[coerced.isnull() & df[col].notnull()].index.tolist()
        if bad:
            logger.warning(f"  [{source}] Non-numeric in '{col}': {len(bad)} records rejected.")
            valid_idx = [i for i in valid_idx if i not in bad]
        df[col] = coerced

    for col, valid_vals in CATEGORICAL_FIELDS.items():
        if col not in df.columns:
            continue
        norm = df[col].astype(str).str.lower().str.strip()
        bad  = norm[~norm.isin(valid_vals) & norm.notnull()].index.tolist()
        if bad:
            logger.warning(f"  [{source}] Invalid category in '{col}': {len(bad)} records.")
            valid_idx = [i for i in valid_idx if i not in bad]
        df[col] = norm

    logger.info(f"[4.1][{source}] Data types enforced. {len(df)-len(valid_idx)} rejected.")
    return df.loc[valid_idx].copy()


def check_range_bounds(df: pd.DataFrame, source: str) -> pd.DataFrame:
    df = df.copy()
    df["range_flag"] = False
    for col, (lo, hi) in RANGE_BOUNDS.items():
        if col not in df.columns:
            continue
        vals = pd.to_numeric(df[col], errors="coerce")
        mask = pd.Series(False, index=df.index)
        if lo is not None: mask |= vals <= lo
        if hi is not None: mask |= vals > hi
        mask &= vals.notnull()
        if mask.sum():
            logger.warning(f"  [{source}] '{col}': {mask.sum()} out-of-range records flagged.")
            df.loc[mask, "range_flag"] = True
    logger.info(f"[4.1][{source}] Range check done. {df['range_flag'].sum()} flagged.")
    return df


# ── 4.2 Completeness & Missingness ────────────────────

RECORD_MISSING_THRESHOLD  = 0.30
FEATURE_MISSING_THRESHOLD = 0.50


def check_record_completeness(df: pd.DataFrame, source: str) -> pd.DataFrame:
    ratio = df.isnull().mean(axis=1)
    bad   = ratio > RECORD_MISSING_THRESHOLD
    logger.info(f"[4.2][{source}] {bad.sum()} records excluded (>{int(RECORD_MISSING_THRESHOLD*100)}% missing).")
    return df[~bad].copy()


def drop_high_missingness_features(df: pd.DataFrame, source: str) -> pd.DataFrame:
    ratio     = df.isnull().mean()
    drop_cols = ratio[ratio > FEATURE_MISSING_THRESHOLD].index.tolist()
    if drop_cols:
        logger.warning(f"[4.2][{source}] Dropping {len(drop_cols)} high-missingness features: {drop_cols}")
        df = df.drop(columns=drop_cols)
    else:
        logger.info(f"[4.2][{source}] No features dropped for missingness.")
    return df


def log_missingness_patterns(df: pd.DataFrame, source: str, group_col: str = None):
    top = df.isnull().mean().sort_values(ascending=False)
    top = top[top > 0].head(8)
    if not top.empty:
        logger.info(f"[4.2][{source}] Top missing features:\n{top.to_string()}")
    if group_col and group_col in df.columns:
        for grp, gdf in df.groupby(group_col):
            high = gdf.isnull().mean()
            high = high[high > 0.10]
            if not high.empty:
                logger.warning(f"  [{source}] Group '{grp}' missingness: {high.index.tolist()}")


# ── 4.3 Consistency Checks ────────────────────────────

INCOME_DEVIATION_THRESHOLD = 0.25


def check_income_vs_employment(df: pd.DataFrame, source: str) -> pd.DataFrame:
    df = df.copy()
    if "employment_status" not in df.columns or "income" not in df.columns:
        return df
    mask = (df["employment_status"] == "unemployed") & (pd.to_numeric(df["income"], errors="coerce") > 0)
    logger.info(f"[4.3][{source}] Income vs employment: {mask.sum()} unemployed with income > 0 flagged.")
    df.loc[mask, "range_flag"] = True
    return df


def check_loan_vs_monthly_payment(df: pd.DataFrame, source: str) -> pd.DataFrame:
    df  = df.copy()
    req = {"loan_amount", "loan_term", "interest_rate", "monthly_payment"}
    if not req.issubset(df.columns) or df[list(req)].isnull().all().any():
        logger.info(f"[4.3][{source}] Loan vs payment check: skipped (fields unavailable).")
        return df
    r        = (pd.to_numeric(df["interest_rate"], errors="coerce") / 100) / 12
    n        = pd.to_numeric(df["loan_term"], errors="coerce")
    P        = pd.to_numeric(df["loan_amount"], errors="coerce")
    computed = np.where(r == 0, P / n, P * (r * (1+r)**n) / ((1+r)**n - 1))
    M        = pd.to_numeric(df["monthly_payment"], errors="coerce")
    dev      = abs(computed - M) / M.replace(0, np.nan)
    mask     = pd.Series(dev > 0.05, index=df.index) & M.notnull()
    logger.info(f"[4.3][{source}] Loan vs monthly payment: {mask.sum()} inconsistent records flagged.")
    df.loc[mask, "range_flag"] = True
    return df


def check_credit_history_vs_age(df: pd.DataFrame, source: str) -> pd.DataFrame:
    df = df.copy()
    if "age" not in df.columns or "credit_history_length" not in df.columns:
        logger.info(f"[4.3][{source}] Credit history vs age: skipped.")
        return df
    age = pd.to_numeric(df["age"], errors="coerce")
    chl = pd.to_numeric(df["credit_history_length"], errors="coerce")
    mask = chl > (age - 18)
    mask &= chl.notnull() & age.notnull()
    logger.info(f"[4.3][{source}] Credit history vs age: {mask.sum()} violations flagged.")
    df.loc[mask, "range_flag"] = True
    return df


def check_plaid_vs_self_reported_income(df: pd.DataFrame, source: str) -> pd.DataFrame:
    df = df.copy()
    if "plaid_income" not in df.columns:
        logger.info(f"[4.3][{source}] Plaid income check: skipped (no Plaid data).")
        return df
    dev  = abs(df["plaid_income"] - df["income"]) / df["income"].replace(0, np.nan)
    mask = dev > INCOME_DEVIATION_THRESHOLD
    logger.info(f"[4.3][{source}] Plaid income mismatch: {mask.sum()} records flagged.")
    df.loc[mask, "range_flag"] = True
    return df


# ── 4.4 Fairness-Aware Checks ─────────────────────────

# Lending Club  → home_ownership as socioeconomic proxy
# German Credit → age_group, foreign_worker, personal_status_sex

LC_DEMOGRAPHIC_COLS = ["home_ownership"]
GC_DEMOGRAPHIC_COLS = ["age_group", "foreign_worker", "personal_status_sex"]

MIN_GROUP_PROPORTION = 0.05
PROXY_CORR_THRESHOLD = 0.70
MAX_DEFAULT_RATE_GAP = 0.15


def _fairness_checks(df: pd.DataFrame, source: str,
                     demo_cols: list, engineered: list):

    # Demographic representation
    for col in demo_cols:
        if col not in df.columns:
            continue
        props = df[col].value_counts(normalize=True)
        under = props[props < MIN_GROUP_PROPORTION]
        if not under.empty:
            logger.warning(f"[4.4][{source}] Under-represented groups in '{col}': {under.to_dict()}")
        else:
            logger.info(f"[4.4][{source}] Representation OK for '{col}'.")

    # Proxy/leakage scan
    for demo_col in demo_cols:
        if demo_col not in df.columns:
            continue
        enc = pd.factorize(df[demo_col])[0]
        for feat in engineered:
            if feat not in df.columns:
                continue
            vals  = pd.to_numeric(df[feat], errors="coerce")
            valid = vals.notnull()
            if valid.sum() < 10:
                continue
            corr = abs(np.corrcoef(enc[valid], vals[valid])[0, 1])
            if corr > PROXY_CORR_THRESHOLD:
                logger.warning(
                    f"[4.4][{source}] Proxy leakage: '{feat}' ↔ '{demo_col}' |r|={corr:.2f}")
            else:
                logger.info(
                    f"[4.4][{source}] Leakage OK: '{feat}' ↔ '{demo_col}' |r|={corr:.2f}")

    # Label distribution by group
    if "default" not in df.columns:
        return
    for col in demo_cols:
        if col not in df.columns:
            continue
        sub   = df[[col, "default"]].dropna()
        rates = sub.groupby(col)["default"].mean()
        gap   = rates.max() - rates.min()
        if gap > MAX_DEFAULT_RATE_GAP:
            logger.warning(
                f"[4.4][{source}] Label bias in '{col}': gap={gap:.2%}\n{rates.to_string()}")
        else:
            logger.info(f"[4.4][{source}] Label distribution OK for '{col}'. Gap={gap:.2%}")


# ════════════════════════════════════════════════════════
#  MASTER PIPELINE  (runs all 4 sections on one dataset)
# ════════════════════════════════════════════════════════

def run_validation_pipeline(df: pd.DataFrame,
                             source: str,
                             demo_cols: list,
                             engineered_features: list) -> pd.DataFrame:

    print(f"\n{'='*58}")
    print(f"  SAAi VALIDATION PIPELINE — {source.upper().replace('_',' ')}")
    print(f"{'='*58}\n")
    logger.info(f"Input: {len(df):,} records, {df.shape[1]} columns.")

    print("── 4.1 Schema Validation ──")
    df = check_required_fields(df, source)
    df = enforce_data_types(df, source)
    df = check_range_bounds(df, source)

    print("\n── 4.2 Completeness & Missingness ──")
    df = check_record_completeness(df, source)
    df = drop_high_missingness_features(df, source)
    log_missingness_patterns(df, source, group_col=demo_cols[0] if demo_cols else None)

    print("\n── 4.3 Consistency Checks ──")
    df = check_income_vs_employment(df, source)
    df = check_loan_vs_monthly_payment(df, source)
    df = check_credit_history_vs_age(df, source)
    df = check_plaid_vs_self_reported_income(df, source)

    print("\n── 4.4 Fairness-Aware Checks ──")
    _fairness_checks(df, source, demo_cols, engineered_features)

    flagged = df.get("range_flag", pd.Series(False, index=df.index)).sum()
    print(f"\n{'='*58}")
    logger.info(f"[{source}] Pipeline complete — {len(df):,} records passed, {flagged:,} flagged.")
    print(f"{'='*58}\n")
    return df


# ════════════════════════════════════════════════════════
#  ENTRY POINT
# ════════════════════════════════════════════════════════

if __name__ == "__main__":

    # ── Lending Club ──────────────────────────────────
    lc_df  = load_lending_club(nrows=50000)
    lc_val = run_validation_pipeline(
        lc_df,
        source="lending_club",
        demo_cols=["home_ownership"],
        engineered_features=["dti", "monthly_payment", "credit_history_length"],
    )

    # ── German Credit ─────────────────────────────────
    gc_df  = load_german_credit()
    gc_val = run_validation_pipeline(
        gc_df,
        source="german_credit",
        demo_cols=["age_group", "foreign_worker", "personal_status_sex"],
        engineered_features=["loan_amount", "loan_term", "credit_history_length"],
    )

    # ── Preview results ───────────────────────────────
    print("\n── Lending Club — Sample (validated) ──")
    lc_cols = [c for c in ["income","loan_amount","dti","fico_score",
                            "employment_status","default","range_flag","home_ownership"]
               if c in lc_val.columns]
    print(lc_val[lc_cols].head(5).to_string())
    print(f"Final shape: {lc_val.shape}\n")

    print("── German Credit — Sample (validated) ──")
    gc_cols = [c for c in ["loan_amount","loan_term","age","age_group",
                            "employment_status","foreign_worker","default","range_flag"]
               if c in gc_val.columns]
    print(gc_val[gc_cols].head(5).to_string())
    print(f"Final shape: {gc_val.shape}")

Using Colab cache for faster access to the 'lending-club' dataset.
Path to dataset files: /kaggle/input/lending-club
Files found: ['/kaggle/input/lending-club/rejected_2007_to_2018Q4.csv.gz', '/kaggle/input/lending-club/accepted_2007_to_2018Q4.csv.gz', '/kaggle/input/lending-club/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv', '/kaggle/input/lending-club/rejected_2007_to_2018q4.csv/rejected_2007_to_2018Q4.csv']
Loading: /kaggle/input/lending-club/accepted_2007_to_2018Q4.csv.gz
Loaded 50,000 records, 151 columns.


  SAAi VALIDATION PIPELINE — LENDING CLUB

INFO: Input: 50,000 records, 152 columns.
── 4.1 Schema Validation ──
INFO: [4.1][lending_club] Required fields: 0 records rejected.
INFO: [4.1][lending_club] Data types enforced. 0 rejected.
INFO: [4.1][lending_club] Range check done. 20 flagged.

── 4.2 Completeness & Missingness ──
INFO: [4.2][lending_club] 29254 records excluded (>30% missing).
INFO: [4.2][lending_club] Top missing features:
mths_since_last_delinq    0.

In [3]:

# ── Save validated datasets for MCP integration pipeline ──

lc_val.to_csv("lending_club_validated.csv", index=False)
gc_val.to_csv("german_credit_validated.csv", index=False)

print(f"lending_club_validated.csv  — {len(lc_val):,} records, {lc_val.shape[1]} columns")
print(f"german_credit_validated.csv — {len(gc_val):,} records, {gc_val.shape[1]} columns")

# ── Also download them directly to your computer ──
from google.colab import files
files.download("lending_club_validated.csv")
files.download("german_credit_validated.csv")

lending_club_validated.csv  — 20,746 records, 110 columns
german_credit_validated.csv — 1,000 records, 23 columns


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>